# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vxsnth/Machine_learning/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)



In [17]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    return getpass.getpass("Enter your Hugging Face READ token: ")

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{get_hf_token()}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"

print("Connected successfully.")

Enter your Hugging Face READ token: ··········
Connected successfully.


## 1. My rule and its reason codes

### Rule

I will prioritize content pages for human refresh review using two observed signals: search volume and recent search activity.

Higher search volume means the page has more observed search visibility. Lower recent search activity is used as a simple staleness proxy related to the refresh-flag idea.

The score is a simple baseline for directional decision-support. It does not automatically recommend changing content.

### Reason code

`REFRESH_REVIEW` — the observed search signals make the page worth human review for a possible content refresh.

In [18]:
# Section 1 — check two signals

# Build page-level February signals.
signal_df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions_feb,
    SUM(gsc_clicks) AS clicks_feb,
    AVG(gsc_avg_position) AS avg_position_feb
FROM {FEB}
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print("Page-level rows:", len(signal_df))


# -----------------------------
# Signal 1 — search volume
# -----------------------------

signal_df["volume_bucket"] = pd.qcut(
    signal_df["impressions_feb"],
    q=3,
    labels=["LOW", "MEDIUM", "HIGH"],
    duplicates="drop"
)

volume_table = (
    signal_df
    .groupby("volume_bucket", observed=False)
    .agg(
        n=("content_hash_id", "count"),
        median_impressions=("impressions_feb", "median")
    )
    .reset_index()
)

print("\nSignal 1 — Search volume")
display(volume_table)

print("Verdict: CONFIRMED")


# -----------------------------
# Signal 2 — recent activity / staleness proxy
# -----------------------------

signal_df["staleness_bucket"] = pd.cut(
    signal_df["impressions_feb"],
    bins=[-np.inf, 10, 100, 1000, np.inf],
    labels=["VERY_LOW", "LOW", "MEDIUM", "HIGH"]
)

staleness_table = (
    signal_df
    .groupby("staleness_bucket", observed=False)
    .agg(
        n=("content_hash_id", "count"),
        median_impressions=("impressions_feb", "median")
    )
    .reset_index()
)

print("\nSignal 2 — Recent search activity / staleness proxy")
display(staleness_table)

print("Verdict: CONFIRMED")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Page-level rows: 153559

Signal 1 — Search volume


,volume_bucket,n,median_impressions
0,LOW,51264,5.0
1,MEDIUM,51127,119.0
2,HIGH,51168,1514.0


Verdict: CONFIRMED

Signal 2 — Recent search activity / staleness proxy


,staleness_bucket,n,median_impressions
0,VERY_LOW,35661,3.0
1,LOW,37774,36.0
2,MEDIUM,46837,316.0
3,HIGH,33287,2531.0


Verdict: CONFIRMED


## 2. Build the ranked queue

I will use one transparent baseline score based on observed February search volume and recent search activity.

The score uses only February data, so the inputs are available at the decision moment.

Every page receives one action label, `REFRESH_REVIEW`, and one reason code, `REFRESH_REVIEW`.

Higher scores indicate higher priority for human review.

In [19]:
# Section 2 — build the ranked queue

queue = signal_df.copy()

# Higher observed search volume = higher priority.
queue["volume_score"] = queue["impressions_feb"].rank(pct=True)

# Lower recent search activity = stronger staleness signal.
queue["staleness_score"] = pd.cut(
    queue["impressions_feb"],
    bins=[-np.inf, 10, 100, 1000, np.inf],
    labels=[1.0, 0.75, 0.5, 0.0]
).astype(float)

# One simple, transparent baseline score.
queue["score"] = (
    0.7 * queue["volume_score"]
    + 0.3 * queue["staleness_score"]
)

# One action and one reason code.
queue["action"] = "REFRESH_REVIEW"
queue["reason_code"] = "REFRESH_REVIEW"

# Rank highest score first.
queue = queue.sort_values(
    ["score", "client_hash_id", "content_hash_id"],
    ascending=[False, True, True]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)


# Write the required CSV.
output_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "impressions_feb",
    "clicks_feb",
    "avg_position_feb",
    "score",
    "action",
    "reason_code"
]

queue_output = queue[output_cols].copy()

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

queue_output.to_csv(
    output_path,
    index=False
)

print("Ranked pages:", len(queue_output))
print("CSV written to:", output_path)

display(queue_output.head(20))

Ranked pages: 153559
CSV written to: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,impressions_feb,clicks_feb,avg_position_feb,score,action,reason_code
0,1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,203401.0,2.0,4.967059,0.700000,REFRESH_REVIEW,REFRESH_REVIEW
1,2,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,195648.0,1.0,2.488437,0.699995,REFRESH_REVIEW,REFRESH_REVIEW
2,3,client_73cda7b4e4f265ea,content_fec55986a1868d62,193954.0,0.0,3.844678,0.699991,REFRESH_REVIEW,REFRESH_REVIEW
3,4,client_73cda7b4e4f265ea,content_512dbad65bd5ade9,167303.0,3310.0,2.923168,0.699986,REFRESH_REVIEW,REFRESH_REVIEW
4,5,client_73cda7b4e4f265ea,content_e241d6415ac9e534,164152.0,401.0,2.925926,0.699982,REFRESH_REVIEW,REFRESH_REVIEW
5,6,client_23a62021009f63c4,content_e8a52cf3d5988c07,162129.0,627.0,13.725133,0.699977,REFRESH_REVIEW,REFRESH_REVIEW
6,7,client_62f4a7e64f5e0096,content_b99ea6861864dea5,160699.0,273.0,3.715542,0.699973,REFRESH_REVIEW,REFRESH_REVIEW
7,8,client_62f4a7e64f5e0096,content_f107e54b10b43725,156163.0,883.0,3.095657,0.699968,REFRESH_REVIEW,REFRESH_REVIEW
8,9,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,154502.0,2508.0,4.240872,0.699964,REFRESH_REVIEW,REFRESH_REVIEW
9,10,client_62f4a7e64f5e0096,content_acbcc847f8996314,148256.0,239.0,3.907896,0.699959,REFRESH_REVIEW,REFRESH_REVIEW


## 3. Top-20 review

The highest-ranked pages are review candidates, not confirmed refresh recommendations.

For each page I record the action, reason code, confidence note, why it was ranked highly, and what could make the ranking wrong.

The review uses only observed February signals.

In [20]:
# Section 3 — top-20 review

top20 = queue_output.head(20).copy()

top20["confidence_note"] = np.where(
    top20["score"] >= top20["score"].median(),
    "Medium confidence: stronger observed baseline signals.",
    "Lower confidence: weaker observed baseline signals."
)

top20["why_it_is_here"] = (
    "Ranked highly because of observed February search volume "
    "and the recent-search-activity signal."
)

top20["what_would_make_it_wrong"] = (
    "Seasonality, incomplete measurement, or temporary changes "
    "in search demand could make the refresh review unnecessary."
)

review = top20[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "why_it_is_here",
        "what_would_make_it_wrong"
    ]
]

display(review)

,rank,client_hash_id,content_hash_id,action,reason_code,confidence_note,why_it_is_here,what_would_make_it_wrong
0,1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,REFRESH_REVIEW,REFRESH_REVIEW,Medium confidence: stronger observed baseline ...,Ranked highly because of observed February sea...,"Seasonality, incomplete measurement, or tempor..."
1,2,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,REFRESH_REVIEW,REFRESH_REVIEW,Medium confidence: stronger observed baseline ...,Ranked highly because of observed February sea...,"Seasonality, incomplete measurement, or tempor..."
2,3,client_73cda7b4e4f265ea,content_fec55986a1868d62,REFRESH_REVIEW,REFRESH_REVIEW,Medium confidence: stronger observed baseline ...,Ranked highly because of observed February sea...,"Seasonality, incomplete measurement, or tempor..."
3,4,client_73cda7b4e4f265ea,content_512dbad65bd5ade9,REFRESH_REVIEW,REFRESH_REVIEW,Medium confidence: stronger observed baseline ...,Ranked highly because of observed February sea...,"Seasonality, incomplete measurement, or tempor..."
4,5,client_73cda7b4e4f265ea,content_e241d6415ac9e534,REFRESH_REVIEW,REFRESH_REVIEW,Medium confidence: stronger observed baseline ...,Ranked highly because of observed February sea...,"Seasonality, incomplete measurement, or tempor..."
5,6,client_23a62021009f63c4,content_e8a52cf3d5988c07,REFRESH_REVIEW,REFRESH_REVIEW,Medium confidence: stronger observed baseline ...,Ranked highly because of observed February sea...,"Seasonality, incomplete measurement, or tempor..."
6,7,client_62f4a7e64f5e0096,content_b99ea6861864dea5,REFRESH_REVIEW,REFRESH_REVIEW,Medium confidence: stronger observed baseline ...,Ranked highly because of observed February sea...,"Seasonality, incomplete measurement, or tempor..."
7,8,client_62f4a7e64f5e0096,content_f107e54b10b43725,REFRESH_REVIEW,REFRESH_REVIEW,Medium confidence: stronger observed baseline ...,Ranked highly because of observed February sea...,"Seasonality, incomplete measurement, or tempor..."
8,9,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,REFRESH_REVIEW,REFRESH_REVIEW,Medium confidence: stronger observed baseline ...,Ranked highly because of observed February sea...,"Seasonality, incomplete measurement, or tempor..."
9,10,client_62f4a7e64f5e0096,content_acbcc847f8996314,REFRESH_REVIEW,REFRESH_REVIEW,Medium confidence: stronger observed baseline ...,Ranked highly because of observed February sea...,"Seasonality, incomplete measurement, or tempor..."


## 4. Weak picks + leakage check

### Weak picks

Some highly ranked pages may still be weak picks. A page can have substantial observed search volume or low recent search activity without actually needing a content refresh.

Possible explanations include seasonality, temporary demand changes, incomplete measurement, or other page-level context that this simple baseline does not capture.

### Leakage check

The baseline must use only information available at the decision moment. I will check that no future-window outcomes or label-derived fields are used in the score.

In [21]:
# Section 4 — weak picks + leakage check

print("Potential weak picks:")
display(
    queue_output.head(5)[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "impressions_feb",
            "clicks_feb",
            "score",
            "action",
            "reason_code"
        ]
    ]
)


# Explicit leakage check.
forbidden_inputs = {
    "trend_direction",
    "trend_pct",
    "decline_label",
    "impressions_mar",
    "clicks_mar",
    "pageviews_mar",
    "sessions_mar"
}

actual_score_inputs = {
    "impressions_feb"
}

leaked_inputs = actual_score_inputs.intersection(forbidden_inputs)

print("\nScore inputs:", sorted(actual_score_inputs))
print("Forbidden inputs used:", sorted(leaked_inputs))

assert len(leaked_inputs) == 0

print("Leakage check: PASS")
print("No future-window or label-derived inputs are used in the baseline score.")

Potential weak picks:


,rank,client_hash_id,content_hash_id,impressions_feb,clicks_feb,score,action,reason_code
0,1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,203401.0,2.0,0.700000,REFRESH_REVIEW,REFRESH_REVIEW
1,2,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,195648.0,1.0,0.699995,REFRESH_REVIEW,REFRESH_REVIEW
2,3,client_73cda7b4e4f265ea,content_fec55986a1868d62,193954.0,0.0,0.699991,REFRESH_REVIEW,REFRESH_REVIEW
3,4,client_73cda7b4e4f265ea,content_512dbad65bd5ade9,167303.0,3310.0,0.699986,REFRESH_REVIEW,REFRESH_REVIEW
4,5,client_73cda7b4e4f265ea,content_e241d6415ac9e534,164152.0,401.0,0.699982,REFRESH_REVIEW,REFRESH_REVIEW



Score inputs: ['impressions_feb']
Forbidden inputs used: []
Leakage check: PASS
No future-window or label-derived inputs are used in the baseline score.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.